<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/miniprojects/MP9.1B/MP9.1B_pump_speed_reference.ipynb)

*Open this notebook in Google Colab with a GPU runtime (Runtime → Change runtime type → T4 GPU). Its first code cell fetches the library files from the public course repository.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# MP9.1B · The Pump's Speed Reference

*Mini project of the course **Deep Learning for Engineering**, Part 2, in one notebook of five parts:*

**A** The Problem · **B** The Data and the Physics · **C** The Reference Model and the Ground Truth · **D** The Physics-Informed Network · **E** The Report

*Parts A to C are given and solved: read them, run them, and use what they build. Parts D and E are yours: the TODO cells.*

In [ ]:
# files-cell (mini project) ----------------------------------------------------
# The library files must sit beside the notebook. Locally they already do. On
# Google Colab they are fetched from the public course repository on every
# run, so a file left over from an earlier session cannot shadow it.
import os, sys, urllib.request
EXERCISE = (['course_core.py', 'pinn_core.py', 'pipe.py'], "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex09.1-cooling-plate/")
PROJECT = (['mp91b.py', 'MP9.1B_truth.npz'], "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/miniprojects/MP9.1B/")
ON_COLAB = "google.colab" in sys.modules
for files, url in (EXERCISE, PROJECT):
    for f in files:
        if ON_COLAB or not os.path.exists(f):
            if not ON_COLAB and os.path.exists(os.path.join("..", "..", "exercises", "Ex09.1-cooling-plate", f)):
                import shutil; shutil.copy(os.path.join("..", "..", "exercises", "Ex09.1-cooling-plate", f), f)
            else:
                urllib.request.urlretrieve(url + f, f)
            print("fetched", f)
for _m in ['course_core', 'pinn_core', 'pipe', 'mp91b']:
    sys.modules.pop(_m, None)
print("files ready")

In [ ]:
# --- setup ---------------------------------------------------------------
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import course_core as cc
import pipe as hs
import mp91b as mp
torch.set_default_dtype(torch.float64)                   # a second derivative of a network needs double precision
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"  # the GPU, when the runtime has one
cc.OUTPUT_DIR = "MP9.1B_outputs"                          # where the report and its figures are written
print("device:", DEVICE, "| torch", torch.__version__)

def middle(fn, reps=5):
    """The middle of five wall-clock timings of fn(), after one warm-up run;
    on a GPU, synchronised so the time is the computation's, not the launch's."""
    sync = torch.cuda.synchronize if torch.cuda.is_available() else (lambda: None)
    fn(); sync(); ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); sync(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts))

---

# Part A · The Problem

## A1 · What is computed, and why an engineer wants it

In a liquid-cooled power converter the coolant's flow is what a controller can change. The power module's losses step up and down with the load - a motor drive accelerating, a charger switching from constant current to constant voltage - and the pump must be told **how much coolant to send at every instant** so that the module never exceeds its temperature limit. Not more: pumping costs electrical energy, wears the pump and makes noise, and a pump run flat out all the time is a pump that fails early.

The question is harder than it looks, because the cold plate stores heat. A controller that only sees the present power raises the flow when the power steps up, but by then the plate is already warm, and the module overshoots. A better schedule **looks ahead**: knowing that a step is coming (a planned acceleration, a scheduled charge), it pre-cools the plate with a little extra flow beforehand, so that the step can be absorbed. Finding that schedule is an **optimal control problem**: the least pumping energy, with the module's temperature at or below its limit throughout.

The answer is a whole function of time, and it changes with every new load profile. A converter that plans its next half hour - or a design study that sweeps many load profiles - asks the question many times. That is where a fast model pays.

This mini project computes the **flow reference for the pump over 2000 s**, for any power step from 300 to 800 W, with one network.

## A2 · The cold plate, and what is new against Ex09.1

The plate is Ex09.1's: a 250 × 130 × 12 mm aluminium plate with the 6 mm copper serpentine, the 50/50 water–glycol coolant of the exercise, supplied at 25 °C. Ex09.1 studied the flow inside one leg; here the tube is a **heat exchanger** and its flow is the **control input**.

**Two changes.** First, the plate is reduced to **one thermal mass**: aluminium conducts so well that, for a controller, the plate has one temperature $T(t)$. The module sits on it with a thermal resistance $R_m$, and the coolant takes heat at a rate $G(Q)\,(T - T_\mathrm{in})$ that grows with the flow $Q$:

$$C\,\frac{dT}{dt} = P(t) - G(Q)\,(T - T_\mathrm{in}), \qquad T(0) = T_\mathrm{in}, \qquad T_\mathrm{module} = T + R_m P(t) \le 80\ ^\circ\mathrm{C}$$

Second, the flow may be **turbulent**. The exercise's 0.3 L/min is laminar; here the pump may run from 0.05 to 4 L/min (Reynolds number 75 to 6000), because a step of 700 W cannot be cooled by laminar flow at all (part A4). The friction and the heat transfer then follow the regime.

The cell below draws the plate and prints its data.

In [ ]:
# The cold plate of Ex09.1, and the data of this project
fig = hs.draw_domain()                                   # Ex09.1's drawing: the plate, the serpentine, the module
plt.show()
rows = [("plate", f"aluminium {1e3*hs.PLATE[0]:.0f} × {1e3*hs.PLATE[1]:.0f} × {1e3*hs.PLATE[2]:.0f} mm: one thermal mass C = {mp.C_PLATE:.0f} J/K"),
        ("tube", f"{1e3*hs.D_TUBE:.0f} mm inside, {mp.TUBE_LEN:.2f} m in the plate (4 legs, 3 bends), inner surface {1e4*mp.A_TUBE:.0f} cm²"),
        ("coolant", f"50/50 water-glycol: ρ = {hs.RHO:.0f} kg/m³, μ = {1e3*hs.MU:.2f} mPa·s, c_p = {mp.CP:.0f} J/(kg·K), k = {mp.K_FLUID:.2f} W/(m·K), Pr = {mp.PR:.1f}"),
        ("supply", f"T_in = {mp.T_IN:.0f} °C"),
        ("module", f"R_m = {mp.R_MODULE} K/W to the plate; limit {mp.T_LIMIT:.0f} °C"),
        ("pump", f"Q = {mp.Q_MIN} to {mp.Q_MAX:.0f} L/min (Re {mp.reynolds(mp.Q_MIN):.0f} to {mp.reynolds(mp.Q_MAX):.0f})"),
        ("power", f"{mp.P_BASE:.0f} W, then a step P_step from {mp.T_UP:.0f} to {mp.T_DOWN:.0f} s, then {mp.P_AFTER:.0f} W to {mp.T_END:.0f} s; P_step = {mp.P_MIN:.0f} to {mp.P_MAX:.0f} W")]
for k, v in rows:
    print(f"{k:>8}  {v}")

**What you should see.** Ex09.1's plate with its serpentine and the module on top, and the table. The plate's heat capacity is 948 J/K, so 1 W heats it by about 1 K per 16 minutes at no flow: the plate is a slow store, and that is what makes looking ahead worth it. Every value is **typical, typed from memory** (as in Ex09.1), and the 80 °C limit, the 0.05 K/W and the 25 °C supply are **assumed**.

## A3 · How the flow takes heat, in both regimes

The coolant warms as it runs through the tube, so the heat it takes per kelvin of the plate's lead over the supply is not $hA$ but

$$G(Q) = \dot m c_p \left(1 - e^{-hA/\dot m c_p}\right), \qquad \dot m = \rho Q$$

(the exchanger's effectiveness for a wall at one temperature). The heat transfer coefficient $h$ depends on the regime:

| regime | Reynolds number | heat transfer | friction (Darcy) | why |
|---|---|---|---|---|
| laminar | below 2300 | Sieder and Tate, $\mathrm{Nu} = 1.86\,(\mathrm{Re}\,\mathrm{Pr}\,D/L)^{1/3}$, at least 3.66 | $f = 64/\mathrm{Re}$ (Ex09.1) | with Pr = 21 the temperature profile needs 0.05 Re Pr D = 2.9 m to develop and the tube is 1 m: the **developing** value, which grows with the flow |
| transition | 2300 to 4000 | a straight blend of the two | a straight blend | no law is reliable here |
| turbulent | above 4000 | Dittus and Boelter, $\mathrm{Nu} = 0.023\,\mathrm{Re}^{0.8}\,\mathrm{Pr}^{0.4}$ | Blasius, $f = 0.316\,\mathrm{Re}^{-1/4}$ | fits to measurements in smooth tubes |

All four correlations are **from memory** (Incropera and DeWitt, ch. 8; White, *Fluid Mechanics*, ch. 6), Dittus–Boelter strictly for Re above 1e4 and used down to 4000 as engineers do. The pump's hydraulic power is $\Delta p\,Q$ with $\Delta p = f\,(L/D)\,\rho\bar u^2/2$ (Ex09.1's `pressure_drop`).

In [ ]:
# The coolant's conductance and the pump's power against the flow
q = np.geomspace(mp.Q_MIN, mp.Q_MAX, 300)                # L/min
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
ax[0].plot(q, mp.conductance(q)); ax[0].set_xscale("log")
ax[0].set_xlabel("flow Q [L/min]"); ax[0].set_ylabel("G, heat taken per kelvin [W/K]")
ax[1].loglog(q, 1e3 * mp.pump_power(q))
ax[1].set_xlabel("flow Q [L/min]"); ax[1].set_ylabel("pump's hydraulic power [mW]")
for a in ax:
    for re in (hs.RE_LAMINAR, hs.RE_TURBULENT):
        a.axvline(re / mp.reynolds(1.0), color="grey", ls=":", lw=0.8)
plt.tight_layout(); plt.show()
for qq in (0.05, 0.3, 1.0, 1.5, 2.0, 3.0, 4.0):
    print(f"{qq:4.2f} L/min: Re {float(mp.reynolds(qq)):5.0f}, G = {float(mp.conductance(qq)):6.2f} W/K, pump {1e3*float(mp.pump_power(qq)):8.2f} mW")

**What you should see.** The conductance rises slowly while the flow is laminar (2.46 W/K at 0.05 L/min, 7.23 at 0.3, 14.48 at 1.5) and then steeply through the transition band (dotted lines, 1.53 and 2.66 L/min) as turbulence takes over: 31.1 W/K at 2 L/min, 86.5 at 4. The pump's power grows with the flow squared while laminar (the pressure drop is proportional to the flow) and faster once turbulent: 2.03 mW at 0.3 L/min, 128 mW at 2, 1219 mW at 4.

## A4 · Why a 700 W step needs turbulent flow

Held steady, the module stays at its limit when $G(Q)\,(T_\mathrm{limit} - R_m P - T_\mathrm{in}) = P$. The cell below finds the least steady flow for each power.

In [ ]:
# The least steady flow that holds the module at 80 °C, against the module's power
G = np.load("MP9.1B_truth.npz")
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.plot(G["steady_power"], G["steady_flow"])
ax.axhline(hs.RE_LAMINAR / mp.reynolds(1.0), color="grey", ls=":", lw=0.8)
ax.axhline(hs.RE_TURBULENT / mp.reynolds(1.0), color="grey", ls=":", lw=0.8)
ax.set_xlabel("module power, steady [W]"); ax.set_ylabel("least flow [L/min]")
plt.tight_layout(); plt.show()
for P in (300, 400, 450, 500, 600, 700, 800):
    qs = float(np.interp(P, G["steady_power"], G["steady_flow"]))
    print(f"{P} W: {qs:.2f} L/min, Re {float(mp.reynolds(qs)):.0f}")

**What you should see.** A curve with a **knee near 500 W**: 0.32 L/min holds 300 W, 0.83 holds 400 W, 1.34 holds 450 W, and then the flow reaches the transition band and the turbulent heat transfer takes over - 1.60 L/min holds 500 W, 2.10 holds 700 W, 2.50 holds 800 W. Laminar flow alone (at most 1.53 L/min) holds about 460 W. A step of 700 W therefore cannot be held with laminar flow: the schedule must turn the flow turbulent, and pay the turbulent pumping power for it.

## A5 · Why there is no exact solution

For a flow held constant the plate's balance is linear in $T$ and has an exact solution, an exponential approach to $T_\mathrm{in} + P/G$; the reference model uses exactly that between its decisions. But the **schedule** $Q(t)$ is the solution of an optimal control problem: minimise $\int \Delta p(Q)\,Q\,dt$ subject to the balance and to $T + R_m P \le 80$ °C at every instant. The cost and the conductance are nonlinear and switch with the regime, and the constraint is a **state constraint** that is active on some intervals and not on others. Its optimality conditions (Pontryagin's) have no closed-form solution here, so the optimal schedule is found **numerically**: part C.

## A6 · What is asked

| quantity | at | required |
|---|---|---|
| the hottest module temperature under your network's flow schedule, integrated exactly | the step powers 350, 550, 650 and 750 W, which you do not train on | at most **80.2 °C** |
| the pump's hydraulic energy over the 2000 s | the same four steps | within **5 %** of the ground truth's (the least possible) |
| the flow before the step | the same four steps | raised in time: above the least flow before the step starts |
| the time to answer a new staircase online | one Colab T4 runtime | at least **10 times shorter** than the reference model's (part C measures it) |

One network, trained once, offline, over the whole range of step powers. Its online answer - the flow schedule for a step power it has not seen - is timed against the reference model, which must solve the optimisation again for every new staircase. The training time is reported apart, as the offline cost.

**Why the tolerances.** Both are the topic's original requirements. 0.2 K over the limit is well below what a junction-temperature estimate resolves; 5 % of the pumping energy is a few milliwatts on average, small against the pump's own losses. The reference model meets both by a wide margin (part C); the network has to meet them on step powers it was not trained on.

---

# Part B · The Data and the Physics

## B1 · What the network takes and returns

The network is the **flow reference**, a function of time and of the step's power:

$$(t,\ P_\mathrm{step})\ \longmapsto\ Q(t;\ P_\mathrm{step}) \quad [\mathrm{L/min}]$$

with $t$ from 0 to 2000 s and $P_\mathrm{step}$ from 300 to 800 W. The step power is an **input**, like time: one training covers every step from 300 to 800 W, and a new staircase is answered by evaluating the network, not by solving again.

**Bounded by its last layer.** The pump can only deliver 0.05 to 4 L/min. Rather than penalising flows outside that range, the last layer maps the network's raw output $z$ into it, for example

$$Q = Q_\mathrm{min}\,\exp\!\big(\ln(Q_\mathrm{max}/Q_\mathrm{min})\,\sigma(z)\big)$$

with $\sigma$ the logistic function: any weights give a flow the pump can deliver, and the logarithmic spacing gives the small laminar flows of the quiet hours as much resolution as the large turbulent ones.

**The pump's command is held.** A pump controller updates its command at a fixed rate. Here the flow is held for 10 s at a time: the network is evaluated in the middle of each 10 s interval, and the plate is integrated exactly over it. The ground truth uses the same 10 s intervals (part C).

## B2 · The physics the network is trained with

There is **no data** to train on: the network learns the schedule from the model alone. For a given $P_\mathrm{step}$, the network's flows $Q_k$ on the 200 intervals are fed to the plate's energy balance, which is integrated **exactly** over each interval (the balance is linear in $T$ for a flow held constant):

$$T_{k+1} = T_{\infty,k} + (T_k - T_{\infty,k})\,e^{-G(Q_k)\,\Delta t/C}, \qquad T_{\infty,k} = T_\mathrm{in} + \frac{P_k}{G(Q_k)}$$

This is a recurrence of differentiable operations, so autograd differentiates the module's temperature at every instant with respect to every weight of the network. The loss has two terms:

| loss term | what it is | what it enforces |
|---|---|---|
| $\mathcal{L}_\mathrm{pump}$ | $\sum_k \Delta p(Q_k)\,Q_k\,\Delta t$, divided by the physics' estimate at that step power | the least pumping energy: the objective |
| $\mathcal{L}_\mathrm{limit}$ | the mean of $\max(0,\ T_\mathrm{module} - 80\ ^\circ\mathrm{C})^2$ over 4 instants of every interval, times a weight $w$ | the limit: a penalty, zero when the limit holds |

both averaged over a batch of step powers. The plate's energy balance is not a loss term here: it is **built in**, integrated exactly inside the loss. (A second network for the plate's temperature, with the balance as a residual, would be the PINN form of the same thing; it is an option, and a harder one to train.)

**The pumping term must be scaled.** The least pumping energy ranges from about 1 J at a 300 W step to about 120 J at 800 W, a factor of a hundred; unscaled, the large steps would decide the training and the small ones would be left with errors of tens of per cent. The physics gives an estimate at each step power without solving anything: the pump's power at the least steady flow for that power (section A4), held for the step's 400 s - 0.95 J at 300 W, 61.21 J at 700 W (`mp.pumping_scale`). Divided by it, every step weighs alike in the loss.

**The penalty's weight decides how close to the limit.** A penalty is zero only when the limit holds, and a finite weight lets the optimiser trade a small excess against a saving in pumping. A larger weight brings the module closer to 80 °C from above, but makes the loss harder to minimise; raising it in stages is a common way out.

In [ ]:
# The model in PyTorch, for the network's loss: the same functions as mp.conductance and mp.pump_power
def T(a):
    '''A NumPy array as an (n, 1) tensor on the device.'''
    return torch.tensor(np.asarray(a, dtype=float), device=DEVICE).reshape(-1, 1)

def conductance_t(q):
    '''W/K, the heat the coolant takes per kelvin, at the flow q (L/min), in torch.'''
    re = q * mp.LMIN / hs.AREA * hs.D_TUBE / hs.NU                         # Reynolds number
    nu_lam = torch.clamp(1.86 * (re * mp.PR * hs.D_TUBE / mp.TUBE_LEN) ** (1 / 3), min=3.66)   # Sieder and Tate
    nu_tur = 0.023 * re ** 0.8 * mp.PR ** 0.4                            # Dittus and Boelter
    w = torch.clamp((re - hs.RE_LAMINAR) / (hs.RE_TURBULENT - hs.RE_LAMINAR), 0.0, 1.0)        # the blend
    hA = ((1 - w) * nu_lam + w * nu_tur) * mp.K_FLUID / hs.D_TUBE * mp.A_TUBE
    mc = q * mp.LMIN * hs.RHO * mp.CP                                     # W/K, the coolant's capacity rate
    return mc * (1 - torch.exp(-hA / mc))

def pump_power_t(q):
    '''W, the pump's hydraulic power at the flow q (L/min), in torch.'''
    qq = q * mp.LMIN
    re = qq / hs.AREA * hs.D_TUBE / hs.NU
    w = torch.clamp((re - hs.RE_LAMINAR) / (hs.RE_TURBULENT - hs.RE_LAMINAR), 0.0, 1.0)
    f = (1 - w) * 64 / re + w * 0.316 * re ** -0.25                      # Darcy's friction factor, both regimes
    u = qq / hs.AREA
    return f * mp.TUBE_LEN / hs.D_TUBE * hs.RHO * u ** 2 / 2 * qq

q = np.array([0.05, 0.3, 1.0, 2.0, 4.0])
print("conductance, torch against NumPy, largest difference:", float(np.abs(conductance_t(T(q)).cpu().numpy().ravel() - mp.conductance(q)).max()), "W/K")
print("pump power, torch against NumPy, largest difference: ", float(np.abs(pump_power_t(T(q)).cpu().numpy().ravel() - mp.pump_power(q)).max()), "W")

**What you should see.** Differences of zero and 3e-17 W, rounding: the torch functions are the model of part A, written so that autograd can differentiate them.

## B3 · What can be built in

* **The bounds** of the flow, by the last layer (section B1).
* **The step's timing.** The step starts at 1200 s and ends at 1600 s for every staircase; features of $t$ that change there - a steep logistic at each edge, or exponentials of the time left before the step - let a small network place the flow's rise and fall where they belong. Before the step the optimal flow rises over several minutes (part C shows how long), so a feature with that time scale helps.
* **What the optimal schedule looks like at the limit.** While the limit is active and the power steady, the least flow is the steady flow of section A4, which the network can learn as a function of $P_\mathrm{step}$.

## B4 · The training points

Each training step draws a batch of step powers from 300 to 800 W (`mp.sample_steps(n, seed)`, uniform). The four **test step powers, 350, 550, 650 and 750 W**, are inside the range, but nothing is fixed at them: the network is scored where it was not trained. The training powers for which the ground truth file also holds a schedule (300, 400, 500, 600, 700 and 800 W) are there for checking, not for training: the network trains on the model alone.

In [ ]:
# A staircase, and the module under three constant flows: why a schedule is needed
tt, _, _ = mp.simulate(np.ones(200), 10.0, 700.0, sub=10)
fig, ax = plt.subplots(1, 2, figsize=(12, 3.4))
ax[0].plot(tt, mp.power(tt - 1e-9, 700.0)); ax[0].set_xlabel("t [s]"); ax[0].set_ylabel("module power, 700 W step [W]")
for qc in (0.3, 2.1, 4.0):
    _, Tm, _ = mp.simulate(np.full(200, qc), 10.0, 700.0, sub=10)
    ax[1].plot(tt, Tm, label=f"{qc} L/min held: pumping {mp.pumping(np.full(200, qc), 10.0):.0f} J, hottest {Tm.max():.1f} °C")
ax[1].axhline(mp.T_LIMIT, color="r", lw=0.6); ax[1].set_xlabel("t [s]"); ax[1].set_ylabel("module temperature [°C]"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

**What you should see.** The staircase, and the module under three constant flows. At 0.3 L/min the step drives the module far over the limit; at 2.1 L/min - the steady flow for 700 W - it stays at or just under it, but pumps 2.1 L/min for the whole 2000 s, about 306 J; at 4 L/min it is far below the limit and the pumping is eight times larger again. The optimal schedule (part C) needs 61.27 J, a fifth of the constant 2.1 L/min.

---

# Part C · The Reference Model and the Ground Truth

## C1 · Two solvers, two jobs

| | the **reference model** | the **ground truth** |
|---|---|---|
| what it is | the optimisation an engineer would run: direct transcription, SLSQP | the same, on a finer time grid, converged |
| decisions | the flow held on 40 s intervals (50 numbers) | on 10 s intervals (200 numbers) |
| the limit checked at | 4 instants per interval | 10 instants per interval (and the result at 50) |
| its job | to be **timed** against the network: the online cost of a new staircase | to **score** the network and the reference |

## C2 · The scheme, step by step

1. **Decisions.** The flow is held constant on each interval of $\Delta t$; the flows $Q_1 \dots Q_n$ are the unknowns, each bounded by the pump's 0.05 and 4 L/min.
2. **The plate, exactly.** For a flow held constant, the balance $C\,dT/dt = P - G(Q)(T - T_\mathrm{in})$ is linear in $T$; over each interval $T$ approaches $T_\infty = T_\mathrm{in} + P/G$ exponentially with the time constant $C/G$. The plate's temperature is propagated interval by interval with that exact solution (`mp.simulate`); no time step error.
3. **The limit, at several instants.** Within an interval the module's temperature is monotone, but a step in power can make it jump; the constraint $T_\mathrm{module} \le 80$ °C is imposed at 4 (reference) or 10 (ground truth) instants per interval.
4. **The objective.** The pump's hydraulic energy $\sum_k \Delta p(Q_k)\,Q_k\,\Delta t$.
5. **The optimiser: SLSQP** (sequential least squares programming, `scipy.optimize.minimize`): at each iteration a quadratic model of the objective and linearised constraints, its gradients by finite differences. It stops when no step improves the objective within the constraints.
6. **The ground truth's start.** It is started from the 20 s solution, repeated on the 10 s intervals; the reference model from a constant 2 L/min, as an engineer facing a new staircase would.

The convergence study refines the intervals: 80, 40, 20 and 10 s. The schedule is piecewise constant, so the pumping energy should approach its limit at first order or better.

In [ ]:
# The ground truth's convergence study at 700 W, from the truth file
print("intervals [s]   pumping [J]   hottest module [°C]   SLSQP time [s]")
for dt, E, pk, sec in zip(G["conv_dt"], G["conv_pumping"], G["conv_peak"], G["conv_seconds"]):
    print(f"{dt:8.0f}        {E:8.3f}        {pk:8.3f}             {sec:6.1f}")
d = np.abs(np.diff(G["conv_pumping"]))
print("change between interval lengths:", " ".join(f"{v:.3f} J" for v in d))
print(f"observed order of the last two halvings {float(G['conv_order']):.2f}; Richardson extrapolation {float(G['conv_extrapolated']):.3f} J; "
      f"10 s intervals off by {100*(G['conv_pumping'][-1]/float(G['conv_extrapolated'])-1):+.2f} %")
print(f"exact integration of the plate against a Runge-Kutta march: {float(G['check_exact_integration']):.1e} K")
for P in G["step_powers"]:
    b = G[f"balance_{int(P)}W"]
    print(f"{int(P)} W: heat in {b[0]:7.0f} J = to the coolant {b[1]:7.0f} J + stored {b[2]:5.0f} J, off by {b[3]:.1e} J; "
          f"pumping {float(G[f'pumping_{int(P)}W']):7.3f} J, hottest {float(G[f'peak_{int(P)}W']):.2f} °C")

**What you should see.** The pumping energy at 700 W falls from 61.71 J on 80 s intervals to 61.59, 61.34 and 61.27 J: finer intervals let the flow follow the limit more closely, so they pump less. The changes, 0.12, 0.25 and 0.07 J, are not monotone at first (on 80 s intervals the step's edges fall on interval boundaries by luck) and then fall by 3.8 (observed order 1.92). The Richardson extrapolation is 61.25 J; the 10 s ground truth is 0.04 % above it, a hundred times inside the 5 % asked. The hottest module is 80.00 °C at every step power: the limit is active, as it must be in a least-pumping schedule. The plate's integration agrees with a Runge–Kutta march to 5e-07 K, and the energy balance (heat in = heat to the coolant + heat stored) closes to 1e-03 J or better at every step power.

In [ ]:
# The ground truth's schedules: the flow and the module's temperature
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
for P in (300, 500, 700, 800):
    ax[0].step(G["t"], G[f"flow_{P}W"], where="post", label=f"{P} W step")
    ax[1].plot(G["t_module"], G[f"module_{P}W"], lw=0.8, label=f"{P} W step")
ax[0].set_yscale("log"); ax[0].set_xlabel("t [s]"); ax[0].set_ylabel("flow [L/min]"); ax[0].legend(fontsize=8)
ax[1].axhline(mp.T_LIMIT, color="r", lw=0.6); ax[1].set_xlabel("t [s]"); ax[1].set_ylabel("module temperature [°C]")
for a in ax:
    a.axvspan(mp.T_UP, mp.T_DOWN, color="0.9", zorder=0)
plt.tight_layout(); plt.show()
q7 = G["flow_700W"]
for t in (600, 900, 1100, 1190, 1200, 1300, 1600):
    print(f"700 W step, t = {t:4d} s: flow {q7[t // 10]:.2f} L/min")

**What you should see.** The shape of a least-pumping schedule. Under the 120 W base the flow sits at the pump's least 0.05 L/min; **minutes before the step it starts to rise** - for the 700 W step, 0.09 L/min at 600 s, 0.19 at 900 s, 0.38 at 1100 s, 0.55 just before the step - so that the plate is cool when the power arrives. During the step the flow jumps to the steady flow of section A4 (2.10 L/min at 700 W, turbulent) and the module sits at 80 °C. After the step it falls back to 0.05 L/min. At 300 W the whole schedule stays laminar, at most 0.32 L/min.

## C3 · The reference model: accuracy and time

The reference model is the same transcription on 40 s intervals, started from a constant 2 L/min. The cell below runs it at the four test step powers, scores it against the ground truth and times it. A new staircase means a new optimisation: that is its online cost. SLSQP runs on the CPU (it is sequential, and its gradients are finite differences); that is where an engineer would run it.

In [ ]:
# The reference model at the four test step powers, against the ground truth
ref = {}
for P in mp.P_TEST:
    r = mp.reference(P)
    E_true = float(G[f"pumping_{int(P)}W"])
    ref[P] = r
    print(f"{P:.0f} W: pumping {r['pumping']:.3f} J (truth {E_true:.3f}, {100*(r['pumping']/E_true-1):+.2f} %), hottest {r['peak']:.2f} °C, "
          f"{r['iterations']} SLSQP iterations, {r['seconds']:.2f} s")
t_ref = middle(lambda: mp.reference(600.0), reps=3)      # the online cost of one new staircase
print(f"\nreference model: {t_ref:.2f} s per new staircase; the network must answer in under {t_ref*1e2:.0f} ms (ten times shorter)")
REF = {"t_ref_s": t_ref, "pumping_ref": {P: ref[P]["pumping"] for P in ref}, "peak_ref": {P: ref[P]["peak"] for P in ref}}   # kept for part E

**What you should see.** At every test step the reference model keeps the module at 80.00 °C and pumps within 0.52 % of the ground truth (2.296, 28.173, 46.504 and 84.966 J against 2.293, 28.319, 46.607 and 84.627 J), well inside the 5 % asked; it can even pump slightly less, because it checks the limit at only 4 instants per interval (checked at 50 afterwards it is still 80.00 °C to two decimals). Each new staircase takes it 0.5 to 1.3 s of SLSQP on the authors' CPU, shared with other jobs (48 to 72 iterations, each with a finite-difference gradient of 50 numbers through the plate's integration); the timed run gives 0.85 s. That is the bar for the network's online time, which part E measures on this same runtime.

## C4 · A controller that does not look ahead

For contrast, a **reactive** controller that sees only the present power: in every 10 s interval it chooses the least flow that brings the module to its limit by the interval's end. It cannot know a step is coming.

In [ ]:
# A reactive controller against the ground truth
for P in (300.0, 550.0, 700.0):
    q, peak = mp.reactive(P)
    print(f"{P:.0f} W step: the reactive controller reaches {peak:.2f} °C ({peak - mp.T_LIMIT:+.2f} K), at most {q.max():.2f} L/min")

**What you should see.** The reactive controller overshoots at every step power: 86.52 °C at 300 W, 98.64 °C at 550 W and 105.97 °C at 700 W, where it runs the pump at the full 4 L/min and still cannot catch up: the plate was too warm when the power rose. **The network's schedule must look ahead**, as the ground truth's does; the requirement "the flow raised before the step" measures that.

---

# Part D · The Physics-Informed Network

## D1 · What is given, and what is yours

**Yours** are three things, each a TODO cell: the **network** (TODO 1), its **training** (TODO 2), and the **figures** you need to understand it (TODO 3). Everything else is given, so that your month goes into the deep learning.

**The contract.** Your network is a PyTorch module called `net`, used as `net(t, P)`:

| argument | meaning | shape |
|---|---|---|
| `t` | time in seconds, 0 to 2000 | `(n, 1)` |
| `P` | the step's power in watts, 300 to 800 | `(n, 1)` |
| returns | the flow reference in L/min, between 0.05 and 4 | `(n, 1)` |

The evaluation calls it at the middle of each 10 s interval, the flow being held over the interval.

## D2 · The loss, given

`rollout(net, P)` takes a batch of step powers, asks the network for the flow on the 200 intervals, and integrates the plate exactly (section B2). It returns the flows, the module's temperature at 4 instants of every interval, and the pumping energy, all differentiable. `losses(net, P, w_limit)` turns them into the two terms of section B2's table, averaged over the batch.

In [ ]:
# The loss, given ---------------------------------------------------------------
DT = 10.0                                                  # s, the flow is held for 10 s
NT = int(mp.T_END / DT)                                    # 200 intervals
TM = torch.tensor((np.arange(NT) + 0.5) * DT, device=DEVICE)   # s, the middle of each interval

def rollout(net, P, sub=4):
    """P: (B,) step powers [W]. Returns the flows (B, NT) [L/min], the module (B, NT*sub) [°C] and the pumping (B,) [J]."""
    B = P.shape[0]
    q = net(TM.repeat(B).reshape(-1, 1), P.repeat_interleave(NT).reshape(-1, 1)).reshape(B, NT)
    Pw = torch.where((TM >= mp.T_UP) & (TM < mp.T_DOWN), P[:, None],
                     torch.where(TM < mp.T_UP, torch.tensor(mp.P_BASE, device=DEVICE), torch.tensor(mp.P_AFTER, device=DEVICE)))
    G = conductance_t(q)                                   # W/K on every interval
    Tinf = mp.T_IN + Pw / G                                # the plate's steady value on every interval
    Tcur = torch.full((B,), mp.T_IN, device=DEVICE)        # the plate starts at the supply's temperature
    tt = torch.tensor(np.linspace(0, DT, sub + 1)[1:], device=DEVICE)
    mods = []
    for k in range(NT):                                    # exact integration, interval by interval
        Ts = Tinf[:, k:k + 1] + (Tcur - Tinf[:, k])[:, None] * torch.exp(-G[:, k:k + 1] * tt / mp.C_PLATE)
        mods.append(Ts + mp.R_MODULE * Pw[:, k:k + 1])     # the module: plate plus R_m P
        Tcur = Ts[:, -1]
    E = (pump_power_t(q) * DT).sum(1)                      # J, the pump's hydraulic energy
    return q, torch.cat(mods, 1), E

def pumping_scale(P):
    """J, the physics' estimate of the least pumping at the step power P: the steady flow's pump power over 400 s."""
    return torch.tensor(mp.pumping_scale(P.detach().cpu().numpy()), device=DEVICE)

def losses(net, P, w_limit):
    q, mod, E = rollout(net, P)
    over = torch.relu(mod - mp.T_LIMIT)                    # K over the limit, zero where it holds
    return {"pump": (E / pumping_scale(P)).mean(),        # the objective, every step power weighted alike
            "limit": w_limit * (over ** 2).mean()}         # the penalty

## D3 · TODO 1 · Your network

Write the class `Net` and make `net = Net().to(DEVICE)`. Decide:

* **what the network sees.** Time and the step power scaled to order one; features of time that carry the step's edges at 1200 and 1600 s and the approach to the step (section B3);
* **its last layer**, which bounds the flow to 0.05 - 4 L/min for any weights (section B1);
* **its size.** Layers, neurons, activation. Remember why Part 2 uses tanh, not ReLU.

What the problem tells you, from parts A to C: the optimal flow spans two decades, from 0.05 L/min in the quiet hours to over 2 L/min during a large step; it jumps at the step's edges; and before the step it rises smoothly over several minutes.

In [ ]:
# TODO 1 - your network. net(t, P) -> Q [L/min], every argument an (n, 1) tensor (see the table above)
import torch.nn as nn

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        ...                                              # your layers

    def forward(self, t, P):
        ...                                              # return the flow, shape (n, 1), within 0.05 - 4 L/min

torch.manual_seed(88)
net = Net().to(DEVICE)
print("weights:", sum(p.numel() for p in net.parameters()))

## D4 · TODO 2 · Your training

Train `net` with `losses`. Decide the optimiser or optimisers, the number of steps, the learning rate, the weight of the limit's penalty and how it changes, the batch of step powers and whether it is drawn afresh. Keep the history of each term for TODO 3, and **measure the training time**: set `TRAIN_SECONDS`, which the evaluation reports as the offline cost.

The course's recipe (Adam on fresh batches, then L-BFGS on one fixed batch) is a good start. `quick_score()` below gives, at the four test step powers, the hottest module and the pumping against the ground truth, in a fraction of a second.

In [ ]:
# A quick score, given: at the test step powers, the hottest module [°C] and the pumping off the ground truth [%]
def quick_score():
    with torch.no_grad():
        q, mod, E = rollout(net, torch.tensor(mp.P_TEST, device=DEVICE), sub=10)
    hot = mod.max(1).values.cpu().numpy()
    off = [100 * (float(e) / float(G[f"pumping_{int(P)}W"]) - 1) for e, P in zip(E, mp.P_TEST)]
    return float(hot.max()), float(np.abs(off).max())

In [ ]:
# TODO 2 - your training
history = {"pump": [], "limit": []}
t0 = time.perf_counter()
...                                                      # your optimiser(s) and loop; append each term to history
TRAIN_SECONDS = time.perf_counter() - t0
hot, off = quick_score()
print(f"trained in {TRAIN_SECONDS:.0f} s; hottest module {hot:.2f} °C, pumping off the truth by up to {off:.1f} %")

## D5 · TODO 3 · Your figures

Draw what you need to understand your network and to answer the report: at least the loss terms during training, and the flow and the module's temperature at one test step power against the ground truth. Save each figure you want in the report as `MP9.1B_report_<name>.png` (in `cc.output_path`): part E embeds them in the PDF.

In [ ]:
# TODO 3 - your figures
...

## D6 · The evaluation, given

This cell asks your trained `net` the questions part E scores, and keeps the answers in the dictionary `R`:

* the flow schedule on the 200 intervals at the four test step powers, at 250 and 850 W outside the range, and at eleven step powers from 300 to 800 W; for each, the module's temperature **integrated exactly with NumPy** (`mp.simulate`, 50 instants per interval, independent of your training code) and the pumping energy;
* the **online time** of a new staircase on this runtime - the flow schedule from one call of `net` - and the same for 10 and 100 staircases at once, against the reference model (SLSQP) run once (its time from part C, `REF`).

Set `TRAIN_SECONDS` to the offline training time you measured; it is reported beside the online times.

In [ ]:
# The evaluation, given ------------------------------------------------------------
net.eval()
def schedule(P):
    # the flow on the 200 intervals [L/min], rows: step powers, from one call of the network
    P = np.atleast_1d(np.asarray(P, dtype=float))
    with torch.no_grad():
        q = net(TM.repeat(len(P)).reshape(-1, 1), T(np.repeat(P, NT)))
    return q.reshape(len(P), NT).cpu().numpy()

steps = {"test": np.array(mp.P_TEST), "outside": np.array([250.0, 850.0]), "sweep": np.linspace(300, 800, 11)}
out = {"t": np.arange(NT) * DT}
for k, Ps in steps.items():
    qs = schedule(Ps)
    out[f"flow_{k}"], out[f"steps_{k}"] = qs, Ps
    out[f"peak_{k}"] = np.array([mp.module_peak(q, DT, P) for q, P in zip(qs, Ps)])      # °C, NumPy, 50 instants per interval
    out[f"pumping_{k}"] = np.array([mp.pumping(q, DT) for q in qs])                     # J
out["module_test"] = np.array([mp.simulate(q, DT, P, sub=10)[1] for q, P in zip(out["flow_test"], mp.P_TEST)])
out["t_module"] = mp.simulate(out["flow_test"][0], DT, mp.P_TEST[0], sub=10)[0]

# online time: the network against the reference model, on this runtime
timing = {}
for k in (1, 10, 100):
    Ps = np.linspace(301.0, 799.0, k)
    timing[f"net_{k}"] = middle(lambda: schedule(Ps))
timing["ref_1"] = REF["t_ref_s"]
out.update({k: np.array(v) for k, v in timing.items()})
out["train_seconds"] = np.array(TRAIN_SECONDS)
out["n_weights"] = np.array(sum(p.numel() for p in net.parameters()))
out["device"] = np.array(DEVICE)
R = out                                                  # part E reads the answers from here
torch.save(net.state_dict(), cc.output_path("MP9.1B_net.pt"))   # the trained weights, beside the report
for k in (1, 10, 100):
    print(f"online, {k:3d} new staircase(s): network {timing[f'net_{k}']*1e3:7.2f} ms")
print(f"reference model, one new staircase: {REF['t_ref_s']*1e3:.0f} ms ({REF['t_ref_s']/timing['net_1']:.0f} times the network's)")
for P, pk, E in zip(mp.P_TEST, out["peak_test"], out["pumping_test"]):
    print(f"{P:.0f} W: hottest {pk:.2f} °C, pumping {E:.3f} J (truth {float(G[f'pumping_{int(P)}W']):.3f} J)")

---

# Part E · The Report

## E1 · What this part does

It scores your network's answers, kept by section D6 in `R`, against the ground truth `G` and the reference model's results `REF` from part C: the goal metrics (TODO 4) and the report's figures (TODO 5). Then come the ten questions.

In [ ]:
# What part E uses, all from the parts above
G = np.load("MP9.1B_truth.npz")
print("in R:", ", ".join(R))
print(f"reference model (part C): {REF['t_ref_s']:.2f} s per new staircase")

## E2 · TODO 4 · The goal metrics

Compute, against the ground truth, and put in the dictionary `METRICS` (a name and a number each; the report prints them as its results table):

* at each test step power: the hottest module (°C), and the pumping energy's difference from the ground truth (per cent);
* at each test step power: the flow 60 s before the step starts, against the least flow (0.05 L/min) and against the ground truth's;
* the network's online time for one new staircase, the reference model's on the same runtime, and their ratio;
* the offline training time.

Then say in one line each whether the four requirements of part A are met.

In [ ]:
# TODO 4 - the goal metrics
METRICS = {}
...
for k, v in METRICS.items():
    print(f"{k:>55}: {v:.2f}")

## E3 · TODO 5 · The report's figures

Draw, and save as `MP9.1B_report_<name>.png` in `cc.output_path`, the figures your answers refer to. At least: the flow schedule and the module's temperature at the four test step powers against the ground truth; the pumping energy against the step power (`R["steps_sweep"]`, `R["pumping_sweep"]`) with the ground truth's points; and the network's online time for 1, 10 and 100 staircases against the reference model's.

In [ ]:
# TODO 5 - the report's figures
...

## E4 · The report

Answer the ten questions below, each in a few sentences **with your numbers**; an answer needs at least fifteen words. Questions 1 to 6 are about your network and its result, 7 to 9 ask you to use it, 10 concludes. The next cell writes `MP9.1B_report.md` with the results table above and your answers, and the cell after it turns the report into a PDF. **Upload the PDF.**

In [ ]:
# TODO: write your answers. Every string below must be replaced.

NAME = "Your Name"

ANSWERS = {
    "1. Your network: its inputs (with any features of time), its last layer and how it bounds the flow, layers, neurons and number of weights. Why this design?": """
""",
    "2. Your loss: the scaling of the pumping term, the weight of the limit's penalty and how it changed during training. Give both terms' values at the end of training.": """
""",
    "3. Your training: optimiser(s), steps, learning rates, batch of step powers, and the offline time on the T4 in seconds.": """
""",
    "4. At the four test step powers: the hottest module under your schedule and the pumping energy against the ground truth, in per cent. Are both requirements met at every step?": """
""",
    "5. The flow before the step: at each test step power, your flow 60 s and 300 s before the step against the ground truth's. When does your flow start to rise, and why must it?": """
""",
    "6. The online time for a new staircase: your network against the reference model on the same runtime, in milliseconds, and the ratio; the same for 10 and 100 staircases. What does each method spend its time on?": """
""",
    "7. During the step, at 550 and 750 W: your flow against the least steady flow of section A4, and the Reynolds number. Which regime does each schedule run in, and what does that cost in pumping?": """
""",
    "8. Outside the training range: your hottest module and pumping at 250 and 850 W against the reference model run there (mp.reference). How does the network fail outside its range?": """
""",
    "9. Train again with another seed. How much do the hottest module and the pumping at the four test steps differ between the two runs? Is your result reproducible within the requirements?": """
""",
    "10. To conclude: for which engineering task would you use your network rather than the reference model, given your measured accuracy, online time and offline time? For which would you not?": """
""",
}

In [ ]:
MIN_WORDS = 15                                              # an answer is a few sentences with numbers, not a word
short = [(i, len(text.split())) for i, text in enumerate(ANSWERS.values(), 1) if len(text.split()) < MIN_WORDS]
if NAME.strip() in ("", "Your Name") or short:
    print("Not ready - the report is written once these are fixed:")
    if NAME.strip() in ("", "Your Name"):
        print("  NAME: write your name")
    for i, n in short:
        print(f"  answer {i}: {n} word(s); write at least {MIN_WORDS}")
else:

    rows = ["| metric | value |", "|---|---|"] + [f"| {k} | {v:.2f} |" if abs(v) >= 0.01 else f"| {k} | {v:.2e} |" for k, v in METRICS.items()]
    out = ["# MP9.1B — The Pump's Speed Reference", "", f"**{NAME}**", "", "Deep Learning for Engineering · Aalborg University · 2026", "",
           "## Results", "", *rows, ""]
    for qq, text in ANSWERS.items():
        out += [f"## {qq}", "", text.strip(), ""]
    path = cc.output_path("MP9.1B_report.md")
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(out))
    print("written", path)

**What you should see.** `Not ready` until every answer is yours; then the path of the report. Then run the cell below: it turns the report into a PDF, with the figures saved as `MP9.1B_report*.png`, and downloads it.

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns MP9.1B_report.md into MP9.1B_report.pdf, with any figure
# saved as MP9.1B_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
pdf_path = os.path.join(cc.OUTPUT_DIR, "MP9.1B_report.pdf")
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md_path = os.path.join(cc.OUTPUT_DIR, "MP9.1B_report.md")
if not os.path.exists(md_path):
    print("No report yet: it is written once every answer above is filled in.\n"
          "Fill them in, run the cell that writes the report, then run this one again.")
else:
    md = open(md_path, encoding="utf-8").read()
    figs = sorted(glob.glob("MP9.1B_report*.png")
                  + glob.glob(os.path.join(cc.OUTPUT_DIR, "MP9.1B_report*.png")))
    if figs:
        imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
        i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
        md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

    html = markdown.markdown(md, extensions=["fenced_code", "tables"])
    css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
    h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
    pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
    weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                           f"<body>{html}</body></html>", base_url=".").write_pdf(pdf_path)
    print("written", pdf_path, f"with {len(figs)} figure(s)" if figs else "")
    try:
        from google.colab import files
        files.download(pdf_path)
    except ImportError:
        pass